In [ ]:
%load_ext autoreload

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import os
import numpy as np
from pathlib import Path
import glob
import h5py

import jax
import jax.numpy as jnp
from jax import vmap, jit, grad, jacfwd

# Optional: higher numeric accuracy (do this before heavy JAX use)
# try:
#     from jax import config as jax_config
#     jax_config.update("jax_enable_x64", True)
# except Exception:
#     pass

import skewerjax as sjx

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('tableau-colorblind10')
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget

In [ ]:
from skewerjax.simulations.simulations import TNGSimulation

In [ ]:
sim = TNGSimulation("/home/STORAGE/TNG50-4")

In [ ]:
# --------------------------------------------
# 1) High-level summary
# --------------------------------------------
print("\n=== Simulation summary ===")
print(sim.summary())

print("\n=== Snapshot numbers discovered ===")
print(sim.snapnums)

print("\n=== Common (stable) cosmology/metadata ===")
for k, v in sim.cosmo.items():
    if v is None:
        continue
    print(f"{k:24s}: {v}")

# --------------------------------------------
# 2) Inspect a particular snapshot
# --------------------------------------------
snapnum = sim.snapnums[len(sim.snapnums)//2]  # pick a middle snapshot, or set snapnum=25
info = sim.get(snapnum)

print(f"\n=== Snapshot {snapnum:03d} basic info ===")
print("snapdir:", info.snapdir)
print("nfiles :", len(info.files))
print("files  :")
for p in info.files[:10]:
    print("  ", p.name)
if len(info.files) > 10:
    print("  ...")

# --------------------------------------------
# 3) Cosmology dictionary for that snapshot
# --------------------------------------------
print(f"\n=== Snapshot {snapnum:03d} parsed cosmology ===")
for k, v in info.cosmo.items():
    if v is None:
        continue
    if isinstance(v, np.ndarray):
        v = v.tolist()
    print(f"{k:24s}: {v}")

# --------------------------------------------
# 4) Raw metadata keys (Header / Parameters / Config)
# --------------------------------------------
print(f"\n=== Snapshot {snapnum:03d} Header keys ===")
print(sorted(info.header.keys()))

print(f"\n=== Snapshot {snapnum:03d} Parameters keys ===")
print(sorted(info.parameters.keys()) if info.parameters else "(none)")

print(f"\n=== Snapshot {snapnum:03d} Config keys ===")
print(sorted(info.config.keys()) if info.config else "(none)")

# Optionally print specific raw header attrs commonly useful
want_header = [
    "BoxSize", "Redshift", "Time", "HubbleParam",
    "Omega0", "OmegaLambda", "OmegaBaryon",
    "NumFilesPerSnapshot", "NumPart_Total", "MassTable",
]
print(f"\n=== Snapshot {snapnum:03d} selected Header values (if present) ===")
for k in want_header:
    if k in info.header:
        v = info.header[k]
        if isinstance(v, np.ndarray):
            v = v.tolist()
        print(f"{k:24s}: {v}")

# --------------------------------------------
# 5) Loop over all snapshots: (snapnum, z, a) quick table
# --------------------------------------------
print("\n=== All snapshots: z and a ===")
print(f"{'snap':>6} {'z':>12} {'a':>12} {'nfiles':>8}")
for n in sim.snapnums:
    c = sim.get(n).cosmo
    z = c.get("Redshift", None)
    a = c.get("ScaleFactor", None)
    nf = len(sim.get(n).files)
    print(f"{n:6d} {str(z):>12} {str(a):>12} {nf:8d}")

In [ ]:
import h5py
from scipy.spatial import cKDTree

## Checkpoint 1 — Understand what a snapshot contains (raw data & structure)

A TNG snapshot is split in multiple files:
Each file contains groups:
- `Header` (metadata: box size, redshift, scale factor, etc.)
- `Parameters` (units + run parameters)
- `PartType0` (gas cells) ✅ what we need for nHI
- other PartTypes (DM, stars, BH...) not needed here

Inside `PartType0`, the _core_ raw arrays we use are:

**Geometry / mesh**
- `Coordinates` : gas cell centers (in code length units, for TNG usually **ckpc/h**)
- `Volume` : cell volumes (in (ckpc/h)3(\mathrm{ckpc}/h)^3(ckpc/h)3)  
    If `Volume` absent, can derive `Volume = Masses / Density`

**Gas properties**
- `Density` : gas mass density in **code units** (not cm−3^{-3}−3 directly)
- `NeutralHydrogenAbundance` (or similarly named) : HI fraction field from the sim
- `StarFormationRate` : identifies star-forming cells (used in correction)

**Why:** fake_spectra needs **positions**, an effective **cell size / smoothing scale**, and the **field value per cell**.

In [ ]:
import numpy as np
import glob
import h5py

base = "/home/STORAGE/TNG50-4"
snapnum = 25
snapdir = f"{base}/snapdir_{snapnum:03d}"
files = sorted(glob.glob(f"{snapdir}/snap_{snapnum:03d}*.hdf5"))
assert files, f"No files found in {snapdir}"

fn0 = files[0]
print("Using:", fn0)

def _attrs_to_dict(h5attrs):
    out = {}
    for k in h5attrs.keys():
        v = h5attrs[k]
        if isinstance(v, np.ndarray) and v.shape == ():
            v = v.item()
        out[k] = v
    return out

with h5py.File(fn0, "r") as f:
    header = _attrs_to_dict(f["Header"].attrs)
    params = _attrs_to_dict(f["Parameters"].attrs) if "Parameters" in f else {}

# ----------------------------
# 1) Basic cosmology/time scalars
# ----------------------------
BoxSize = float(header["BoxSize"])
z = float(header["Redshift"])
a = float(header["Time"])
h = float(header["HubbleParam"])

print("\n=== Header scalars ===")
print("BoxSize        =", BoxSize)
print("Redshift z     =", z)
print("Scale factor a =", a)
print("HubbleParam h  =", h)

# sanity check: a ~ 1/(1+z)
a_check = 1.0 / (1.0 + z)
print("\nCheck a vs 1/(1+z):")
print("a(header)      =", a)
print("1/(1+z)        =", a_check)
print("relative diff  =", abs(a - a_check) / a_check)

# ----------------------------
# 2) Unit scalars
# ----------------------------
print("\n=== Unit scalars (Parameters) ===")
UnitLength_in_cm = params.get("UnitLength_in_cm", None)
UnitMass_in_g = params.get("UnitMass_in_g", None)
UnitVelocity_in_cm_per_s = params.get("UnitVelocity_in_cm_per_s", None)
UnitDensity_in_cgs = params.get("UnitDensity_in_cgs", None)

print("UnitLength_in_cm          =", UnitLength_in_cm)
print("UnitMass_in_g             =", UnitMass_in_g)
print("UnitVelocity_in_cm_per_s  =", UnitVelocity_in_cm_per_s)
print("UnitDensity_in_cgs (given)=", UnitDensity_in_cgs)

# derive UnitDensity_in_cgs if needed
if UnitDensity_in_cgs is None and UnitMass_in_g is not None and UnitLength_in_cm is not None:
    UnitDensity_in_cgs = float(UnitMass_in_g) / (float(UnitLength_in_cm) ** 3)
    print("UnitDensity_in_cgs (derived) =", UnitDensity_in_cgs)

# if both exist, check consistency
if ("UnitDensity_in_cgs" in params) and (UnitMass_in_g is not None) and (UnitLength_in_cm is not None):
    derived = float(UnitMass_in_g) / (float(UnitLength_in_cm) ** 3)
    given = float(params["UnitDensity_in_cgs"])
    print("\nCheck UnitDensity_in_cgs consistency:")
    print("given   =", given)
    print("derived =", derived)
    print("relative diff =", abs(given - derived) / given)

# ----------------------------
# 3) Interpret BoxSize units (TNG convention: ckpc/h)
# ----------------------------
L_box_cMpc_over_h = BoxSize / 1000.0         # if BoxSize is ckpc/h
L_box_cMpc = BoxSize / (1000.0 * h)          # comoving Mpc (not /h)

print("\n=== BoxSize interpreted as ckpc/h ===")
print("L_box [cMpc/h] =", L_box_cMpc_over_h)
print("L_box [cMpc]   =", L_box_cMpc)

# Useful for later: conversion factor coords_ckpc_over_h -> coords_cMpc
print("\nConversion: ckpc/h -> cMpc is divide by (1000*h) =", 1.0/(1000.0*h))
